# Лабораторная 3. Аудит собственного оценщика
**П3 · 17 ноября 2026 · индивидуальная часть — 35 минут**

Сначала разберём три измерителя на небольших синтетических примерах. Затем команда применит протокол к своему срезу.

**Как работать:** меняйте только отмеченные ячейки `EDIT HERE`. Остальные ячейки нужны, чтобы примеры, данные и формат файлов оставались согласованными. В конце ноутбука вы подготовите командные файлы для сдачи. После сдачи кураторы проверят командные файлы и вернут командам результаты.

## Как работать с ноутбуком

`EDIT HERE` — здесь вы дописываете решение или меняете значения в настройках команды.

`LOCKED` — эти ячейки содержат исходные данные, заготовки и код, который связывает задания между собой. Их лучше не редактировать.

**В этой лабораторной вы:**
- реализуете три небольшие функции для анализа согласия и простой базовой линии;
- проверите их на учебных синтетических данных;
- подготовите командные файлы на основе среза, который команда самостоятельно подготовит до занятия по шаблонам;
- сохраните свою работу и командные файлы для сдачи.

**Важно:** встроенные тесты ниже дают быструю обратную связь по трём учебным функциям. Это не итоговая проверка командной работы и не выставление оценки. После сдачи кураторы отдельно проверят командные файлы.

Не меняйте имена функций, аргументы, формат возвращаемых значений, seed и ID объектов. В `annotations.csv` заполняйте только `label` и `elapsed_seconds`; порядок предъявления и служебные поля уже подготовлены.

In [2]:
# LOCKED — импорты и фиксированные зависимости
import numpy as np
from collections import Counter, defaultdict
import math, csv, json, random
from pathlib import Path


In [3]:
# LOCKED — синтетическая фикстура, seed и ожидаемые эффекты
SEED = 2026

def make_fixture(seed=SEED):
    rng = np.random.default_rng(seed)
    truth = rng.integers(0, 2, size=20)
    labels = np.tile(truth[:, None], (1, 4))
    for j, flip_rate in enumerate([0.10, 0.15, 0.20, 0.10]):
        flips = rng.random(20) < flip_rate
        labels[:, j] = np.where(flips, 1-truth, truth)

    first_wins = np.array([True]*78 + [False]*42)
    rng.shuffle(first_wins)
    judgments = [{"id":f"J{i+1:03d}", "first_won":bool(v),
                  "order":"A-first" if v else "B-first"}
                 for i,v in enumerate(first_wins)]

    directions = {}
    for direction in ["math","code","labeling","logs","environments"]:
        objects, humans = [], []
        for i in range(12):
            objects.append({
                "id":f"{direction[:3].upper()}-{i+1:02d}",
                "length_a":int(rng.integers(20,100)),
                "length_b":int(rng.integers(20,100)),
                "votes_a":int(rng.integers(0,5)),
                "votes_b":int(rng.integers(0,5)),
                "tests_a":bool(rng.integers(0,2)),
                "tests_b":bool(rng.integers(0,2)),
                "terminal_a":bool(rng.integers(0,2)),
                "terminal_b":bool(rng.integers(0,2)),
                "action_count_a":int(rng.integers(0,10)),
                "action_count_b":int(rng.integers(0,10)),
            })
            latent = "A" if rng.random()<0.5 else "B"
            humans.append([latent if rng.random()>0.20 else ("B" if latent=="A" else "A")
                           for _ in range(4)])
        directions[direction]={"objects":objects,"humans":humans}
    return {"core_labels":labels,"judgments":judgments,"directions":directions}

fixture = make_fixture()
print("Фикстура загружена:", fixture["core_labels"].shape, "и", len(fixture["judgments"]), "парных суждений")


Фикстура загружена: (20, 4) и 120 парных суждений


## Что именно считаем

1. `agreement(labels)` — среднюю попарную долю совпадений, номинальную α Криппендорфа и bootstrap-интервалы.
2. `first_shown_rate(judgments)` — долю случаев, когда победил первый показанный вариант, и интервал Уилсона.
3. `trivial_scorer(objects, rule)` и `vs_humans(verdicts, humans)` — простую базовую линию и её согласие с человеческим большинством.

На учебных примерах мы проверяем, что измерители считают именно то, что задумано. В командной части вы примените протокол к своему материалу. При двух разметчиках считаем процент совпадений; α Криппендорфа в таком случае не интерпретируем.

## Задание 1 — согласие людей · `agreement(labels)`

Реализуйте среднюю попарную долю совпадений, номинальную α Криппендорфа и 95%-й bootstrap-интервал по объектам.

Верните словарь с ключами `pairwise_agreement`, `alpha`, `agreement_ci`, `alpha_ci`, `unanimous_share`.


In [4]:
# EDIT HERE — не меняйте имя функции, аргументы и ключи результата
def agreement(labels, n_boot=400, seed=7):
    def _pairwise_one(row):
        m = len(row)
        agree = pairs = 0
        for i in range(m):
            for j in range(i + 1, m):
                pairs += 1
                if row[i] == row[j]:
                    agree += 1
        return agree / pairs if pairs else 1.0

    def _value_counts(labels_arr):
        cats = np.unique(labels_arr)
        cat_idx = {c: i for i, c in enumerate(cats)}
        vc = np.zeros((labels_arr.shape[0], len(cats)), dtype=np.int64)
        for ui, row in enumerate(labels_arr):
            for v in row:
                vc[ui, cat_idx[v]] += 1
        return vc

    def _coincidence_matrix(value_counts):
        pairable = np.maximum(value_counts.sum(axis=1), 2).astype(float)
        V = value_counts.shape[1]
        diagonals = value_counts[:, np.newaxis, :] * np.eye(V)[np.newaxis, ...]
        unnorm = (
            value_counts[..., np.newaxis] * value_counts[:, np.newaxis, :]
            - diagonals
        )
        return np.divide(
            unnorm,
            (pairable - 1).reshape((-1, 1, 1)),
            where=(pairable[:, None, None] > 1),
            out=np.zeros_like(unnorm, dtype=float),
        ).sum(axis=0)

    def _random_coincidences(n_v):
        total = n_v.sum()
        if total <= 1:
            return np.zeros((len(n_v), len(n_v)), dtype=float)
        return np.divide(
            np.outer(n_v, n_v) - np.diagflat(n_v),
            total - 1,
            dtype=float,
        )

    def _nominal_alpha(labels_arr):
        o = _coincidence_matrix(_value_counts(labels_arr))
        n_v = o.sum(axis=0)
        e = _random_coincidences(n_v)
        o_dis = o.sum() - np.trace(o)
        e_dis = e.sum() - np.trace(e)
        if e_dis == 0:
            return 1.0
        return 1.0 - o_dis / e_dis

    n_objects = labels.shape[0]
    pairwise_agreement = float(
        np.mean([_pairwise_one(labels[k]) for k in range(n_objects)])
    )
    unanimous_share = float(
        np.mean([len(set(labels[k].tolist())) == 1 for k in range(n_objects)])
    )
    alpha = float(_nominal_alpha(labels))

    rng = np.random.default_rng(seed)
    boot_agree, boot_alpha = [], []
    idx = np.arange(n_objects)
    for _ in range(n_boot):
        sample = rng.choice(idx, size=n_objects, replace=True)
        sub = labels[sample]
        boot_agree.append(float(np.mean([_pairwise_one(sub[k]) for k in range(n_objects)])))
        boot_alpha.append(float(_nominal_alpha(sub)))

    agreement_ci = (
        float(np.percentile(boot_agree, 2.5)),
        float(np.percentile(boot_agree, 97.5)),
    )
    alpha_ci = (
        float(np.percentile(boot_alpha, 2.5)),
        float(np.percentile(boot_alpha, 97.5)),
    )

    return {
        "pairwise_agreement": pairwise_agreement,
        "alpha": alpha,
        "agreement_ci": agreement_ci,
        "alpha_ci": alpha_ci,
        "unanimous_share": unanimous_share,
    }


## Задание 2 — позиционное смещение · `first_shown_rate(judgments)`

Посчитайте долю побед первого показанного варианта и 95%-й интервал Уилсона.

Верните словарь с ключами `wins`, `n`, `rate`, `ci`.


In [5]:
# EDIT HERE — не меняйте имя функции, аргументы и ключи результата
def first_shown_rate(judgments):
    def _wilson_ci(wins, n, z=1.96):
        if n == 0:
            return (0.0, 0.0)
        p = wins / n
        denom = 1 + z**2 / n
        centre = (p + z**2 / (2 * n)) / denom
        margin = (z / denom) * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
        return (max(0.0, centre - margin), min(1.0, centre + margin))

    n = len(judgments)
    wins = sum(1 for j in judgments if j["first_won"])
    rate = wins / n if n else 0.0
    ci = _wilson_ci(wins, n)
    return {"wins": wins, "n": n, "rate": rate, "ci": ci}


## Задание 3 — тривиальная нижняя линия · `trivial_scorer` и `vs_humans`

Реализуйте пять правил для синтетической фикстуры:
- `majority`: больше голосов за A или B;
- `tests`: проходит ли тесты A или B;
- `longer`: какой вариант длиннее;
- `most_common_action`: у какого действия больше наблюдений;
- `terminal`: какой вариант достиг терминального состояния.

При равенстве верните `TIE`. `vs_humans` сравнивает вердикт с мажоритарной меткой; объекты с ничьей в человеческой разметке исключаются.


In [6]:
# EDIT HERE — сохраните сигнатуры и формат результата
def trivial_scorer(objects, rule):
    verdicts = []
    for obj in objects:
        if rule == "majority":
            va, vb = obj["votes_a"], obj["votes_b"]
        elif rule == "tests":
            va, vb = obj["tests_a"], obj["tests_b"]
        elif rule == "longer":
            va, vb = obj["length_a"], obj["length_b"]
        elif rule == "most_common_action":
            va, vb = obj["action_count_a"], obj["action_count_b"]
        elif rule == "terminal":
            va, vb = obj["terminal_a"], obj["terminal_b"]
        else:
            raise ValueError(f"Unknown rule: {rule}")
        if va == vb:
            verdicts.append("TIE")
        elif va > vb:
            verdicts.append("A")
        else:
            verdicts.append("B")
    return verdicts

def vs_humans(verdicts, humans):
    def _wilson_ci(wins, n, z=1.96):
        if n == 0:
            return (0.0, 0.0)
        p = wins / n
        denom = 1 + z**2 / n
        centre = (p + z**2 / (2 * n)) / denom
        margin = (z / denom) * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
        return (max(0.0, centre - margin), min(1.0, centre + margin))

    matches = 0
    n = 0
    for verdict, votes in zip(verdicts, humans):
        cnt = Counter(votes)
        top = cnt.most_common()
        if len(top) >= 2 and top[0][1] == top[1][1]:
            continue
        majority = top[0][0]
        n += 1
        if verdict == majority:
            matches += 1
    agreement = matches / n if n else 0.0
    ci = _wilson_ci(matches, n)
    return {"agreement": agreement, "matches": matches, "n": n, "ci": ci}


## Учебные тесты для трёх функций — запустите после реализации

Следующая ячейка проверяет ваши функции на фиксированных синтетических примерах. Это небольшая встроенная проверка, которая помогает заметить ошибку в реализации и вернуться к нужной функции.

Она **не проверяет командную сдачу**, не создаёт итоговый отчёт и не заменяет проверку кураторов. Если часть тестов не проходит, посмотрите на сообщение, проверьте соответствующую функцию и попробуйте ещё раз. Можно также попросить куратора объяснить непонятный тест.

In [7]:
# LOCKED — учебные тесты. Не редактировать.
def _practice_tests():
    results = []
    def check(name, condition, got=None):
        results.append(bool(condition))
        print(("OK" if condition else "НУЖНО ПОСМОТРЕТЬ") + " — " + name +
              ("" if got is None else f" (получено: {got})"))

    try:
        a = agreement(fixture["core_labels"])
        check("средняя попарная доля", abs(a["pairwise_agreement"]-0.8916666666666666)<1e-6, round(a["pairwise_agreement"],4))
        check("альфа Криппендорфа", abs(a["alpha"]-0.7792821835375027)<1e-6, round(a["alpha"],4))
        check("интервалы рассчитаны", len(a["agreement_ci"])==2 and len(a["alpha_ci"])==2)
        check("доля полного согласия в пределах 0–1", 0 <= a["unanimous_share"] <= 1, round(a["unanimous_share"],4))
    except Exception as e:
        for name in ["средняя попарная доля","альфа Криппендорфа","интервалы рассчитаны","доля полного согласия в пределах 0–1"]:
            if len(results)<4: check(name,False,repr(e))

    try:
        f = first_shown_rate(fixture["judgments"])
        check("число побед первого варианта", f["wins"]==78 and f["n"]==120 and abs(f["rate"]-.65)<1e-9, f'{f["wins"]}/{f["n"]}')
        check("интервал Уилсона рассчитан", len(f["ci"])==2 and 0<=f["ci"][0]<=f["rate"]<=f["ci"][1]<=1, f["ci"])
    except Exception as e:
        check("позиционное смещение",False,repr(e))

    rules={"math":"majority","code":"tests","labeling":"longer","logs":"most_common_action","environments":"terminal"}
    expected={"math":5/9,"code":2/9,"labeling":6/10,"logs":7/11,"environments":3/11}
    for d,rule in rules.items():
        try:
            block=fixture["directions"][d]
            pred=trivial_scorer(block["objects"],rule)
            out=vs_humans(pred,block["humans"])
            check(f"базовая линия: {d}", len(pred)==12 and abs(out["agreement"]-expected[d])<1e-9 and len(out["ci"])==2, f'{out["matches"]}/{out["n"]}')
        except Exception as e:
            check(f"базовая линия: {d}",False,repr(e))

    print(f"\nУчебные тесты пройдены: {sum(results)}/{len(results)}")
    print("Эти результаты относятся только к учебным функциям на синтетической фикстуре.")
    return all(results)

PRACTICE_TESTS_PASSED = _practice_tests()

OK — средняя попарная доля (получено: 0.8917)
OK — альфа Криппендорфа (получено: 0.7793)
OK — интервалы рассчитаны
OK — доля полного согласия в пределах 0–1 (получено: 0.8)
OK — число побед первого варианта (получено: 78/120)
OK — интервал Уилсона рассчитан (получено: (0.5612115776488139, 0.7294823410344072))
OK — базовая линия: math (получено: 5/9)
OK — базовая линия: code (получено: 2/9)
OK — базовая линия: labeling (получено: 6/10)
OK — базовая линия: logs (получено: 7/11)
OK — базовая линия: environments (получено: 3/11)

Учебные тесты пройдены: 11/11
Эти результаты относятся только к учебным функциям на синтетической фикстуре.


## TEAM CONFIG — настройки командной части

Заполните эту ячейку после индивидуальных упражнений. Используйте псевдонимы участников (A1–A4), а не полные имена. До запуска генератора убедитесь, что в папке `audit/` лежат файлы `slice.jsonl` и `instruction.md`, которые команда подготовила до занятия. Для дорожки A `scorer.jsonl` готовится и хранится отдельно у ответственного, который не участвует в разметке, или у куратора. После завершения человеческой разметки его можно добавить в `audit/` для сдачи.

In [ ]:
# EDIT HERE — измените только значения справа от знака =
TEAM_ID = "K02"                 # K01 ... K16
DIRECTION = "environments"              # code / math / labeling / environments / logs
ANNOTATOR_IDS = ["A1","A2","A3","A4"]  # фактически участвующие разметчики (2–4)
PAIRWISE_TASK = False             # True, если инструкция просит сравнивать A/B
AUDIT_DIR = "audit"              # папка с входными файлами, подготовленными командой

Path(AUDIT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Рабочая папка: {Path(AUDIT_DIR).resolve()}")
print("Проверьте, что команда положила сюда slice.jsonl и instruction.md.")

## Подготовка командных CSV-файлов

Запустите следующую ячейку после заполнения `TEAM CONFIG` и проверки входных файлов. Она создаст три заготовки:
- `annotations.csv` — таблицу для человеческих суждений;
- `baseline_predictions.csv` — таблицу для вердиктов простой базовой линии;
- `probe_results.csv` — таблицу для записи проверок-зондов.

В `annotations.csv` заполняйте только `label` и `elapsed_seconds`. Поля `id`, `annotator`, `order` и названия столбцов уже подготовлены. В `baseline_predictions.csv` заполните `verdict` для каждого ID. В `probe_results.csv` добавьте результаты минимум двух зондов для дорожки A или одного для дорожки B. В столбце `n` укажите целое число больше нуля, в `value` — численный результат без знака `%` (например, 0.65 вместо 65%).

Команда заранее кладёт в `audit/` `slice.jsonl` и `instruction.md`. Для дорожки A `scorer.jsonl` готовит ответственный, который не участвует в человеческой разметке, либо файл хранит куратор. Участники-разметчики не открывают его до завершения всей человеческой разметки; после этого файл добавляют в `audit/` для сдачи.

In [8]:
# LOCKED — генератор командных шаблонов

def create_annotations_template(audit_dir, annotator_ids, pairwise=False, seed=2026):
    root=Path(audit_dir)
    rows=[json.loads(x) for x in (root/"slice.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
    core=[x["id"] for x in rows if x["group"]=="core"]
    second=[x["id"] for x in rows if x["group"]=="second"]
    if len(core)!=20 or len(second)!=32:
        raise ValueError("В предоставленном срезе должно быть 20 объектов core и 32 объекта second")
    if not 2 <= len(annotator_ids) <= 4:
        raise ValueError("Укажите от 2 до 4 фактически участвующих разметчиков")
    rng=random.Random(seed)
    rng.shuffle(second)
    out=[]
    for person in annotator_ids:
        for object_id in core:
            out.append({"id":object_id,"annotator":person,"label":"",
                        "order":rng.choice(["A-first","B-first"]) if pairwise else "NA",
                        "instruction_version":"1.0","elapsed_seconds":""})
    for j, person in enumerate(annotator_ids):
        for object_id in second[j*8:(j+1)*8]:
            out.append({"id":object_id,"annotator":person,"label":"",
                        "order":rng.choice(["A-first","B-first"]) if pairwise else "NA",
                        "instruction_version":"1.0","elapsed_seconds":""})
    path=root/"annotations.csv"
    with path.open("w",encoding="utf-8-sig",newline="") as f:
        writer=csv.DictWriter(f,fieldnames=["id","annotator","label","order","instruction_version","elapsed_seconds"])
        writer.writeheader(); writer.writerows(out)
    with (root/"baseline_predictions.csv").open("w",encoding="utf-8-sig",newline="") as f:
        w=csv.writer(f); w.writerow(["id","verdict"])
        for x in rows: w.writerow([x["id"],""])
    with (root/"probe_results.csv").open("w",encoding="utf-8-sig",newline="") as f:
        csv.writer(f).writerow(["probe","n","metric","value","interpretation"])
    print("Созданы: annotations.csv, baseline_predictions.csv, probe_results.csv")
    print("Заполните метки и результаты по инструкции; исходные ID и заголовки оставьте без изменений.")
    return path

# Запустите после проверки папки audit:
# create_annotations_template(AUDIT_DIR, ANNOTATOR_IDS, PAIRWISE_TASK)

## Что сохранить и отправить после занятия

**Каждый участник:** сохраните свою копию этого ноутбука с именем `Kxx_A1_lab3_student.ipynb` (вместо `A1` используйте свой псевдоним) и положите её в папку команды `notebooks/`.

**Одна общая папка `audit/` на команду** должна содержать:
- `slice.jsonl` и `instruction.md` — подготовлены командой до занятия; не удаляйте и не переименовывайте;
- `annotations.csv` — создан в этом ноутбуке, заполнен командой;
- `baseline_predictions.csv` — создан в этом ноутбуке, заполнен командой;
- `probe_results.csv` — создан в этом ноутбуке, заполнен командой;
- `scorer.jsonl` — только для дорожки A; до окончания разметки хранится отдельно у ответственного, не участвующего в разметке, или у куратора; после разметки добавляется в `audit/`.

**Дополнительные служебные отчёты создавать не нужно.** После сдачи кураторы отдельно проверят командные файлы и сообщат, если потребуется что-то уточнить.

Одна команда отправляет одну ссылку на общую папку через Google Form. Внутри папки сохраните структуру `audit/` и `notebooks/`, чтобы куратору было удобно найти файлы.